# 02 — Synthetic Error Insertion on FinQA (First 5 Rows)

**Goal:** Use the Groq API to insert exactly one synthetic factual error  
into the `response` field of the first 5 rows of the FinQA training split.

**Error types:** Temporal · Numerical · Entity · Relation · Contradictory · Unverifiable

**Tag format:**
- Span-level: `<type><delete>original_span</delete><mark>corrupted_span</mark></type>`
- Sentence-level (Contradictory / Unverifiable): `<type>corrupted_sentence</type>`


In [ ]:
# ── Install dependencies if needed ────────────────────────────────────────
# %pip install -q datasets groq python-dotenv

In [ ]:
import os, requests
from dotenv import load_dotenv, find_dotenv
from datasets import load_dataset
from groq import Groq

# ── Load API key from .env ────────────────────────────────────────────────
env_file = find_dotenv(usecwd=True)
loaded   = load_dotenv(env_file, override=True)
print(f".env path : {env_file or 'NOT FOUND'}")
print(f"loaded    : {loaded}")

GROQ_API_KEY = os.environ.get("GROQ_API_KEY", "")
if not GROQ_API_KEY:
    raise ValueError("GROQ_API_KEY not found in .env")

# ── Model priority list ───────────────────────────────────────────────────
PREFERRED = [
    "qwen/qwen3.6-27b",
    "qwen/qwen3.8-27b",
    "openai/gpt-oss-20b",
    "openai/gpt-oss-120b",
]

resp      = requests.get(
    "https://api.groq.com/openai/v1/models",
    headers={"Authorization": f"Bearer {GROQ_API_KEY}"},
)
available = {m["id"] for m in resp.json().get("data", [])}

MODEL = next((m for m in PREFERRED if m in available), None)
if not MODEL:
    text_models = [
        m for m in sorted(available)
        if not any(x in m for x in ["whisper", "guard", "orpheus"])
    ]
    MODEL = text_models[0] if text_models else None
    if not MODEL:
        raise ValueError(f"No usable text-generation model found. Available: {available}")

N_ROWS = 5
client = Groq(api_key=GROQ_API_KEY)
print(f"\n✅ Using model: {MODEL}")

In [ ]:
# ── Load dataset ──────────────────────────────────────────────────────────
ds    = load_dataset("rungalileo/ragbench", "finqa", trust_remote_code=True)
train = ds["train"]

print(f"FinQA train split: {len(train):,} rows")
print(f"Columns          : {train.column_names}")

In [ ]:
# ── Prompt template ───────────────────────────────────────────────────────
PROMPT_TEMPLATE = """\
You are a dataset-annotation assistant. Your task is to insert exactly ONE factual error \
into the RESPONSE below. Use the reference DOCUMENTS and QUESTION only to understand context \
— do NOT alter them.

Choose ONE error type at random from this list:
  Temporal, Numerical, Entity, Relation, Contradictory, Unverifiable

Tagging rules (follow exactly, no exceptions):
  • Span-level errors (Temporal / Numerical / Entity / Relation):
      <TYPE><delete>original_span</delete><mark>corrupted_span</mark></TYPE>
    Replace TYPE with the chosen error type in the exact casing shown above.
  • Sentence-level errors (Contradictory / Unverifiable):
      <TYPE>corrupted_sentence</TYPE>
    The corrupted sentence must replace the original sentence entirely inside the tag.

Constraints:
  - Insert EXACTLY one error. No more.
  - Output ONLY the tagged corrupted response. No explanation, no preamble.
  - Do NOT change any part of the response that is not involved in the error.
  - Do NOT add new sentences. Only corrupt or replace an existing span or sentence.

---
DOCUMENTS:
{documents}

QUESTION:
{question}

RESPONSE:
{response}
---

Output the tagged corrupted response now:"""

print("Prompt template defined ✓")

In [ ]:
# ── Helpers ───────────────────────────────────────────────────────────────
def format_documents(docs):
    if isinstance(docs, list):
        return "\n\n".join(f"[Doc {i+1}] {d}" for i, d in enumerate(docs))
    return str(docs)


def insert_error(documents_raw, question, response):
    prompt = PROMPT_TEMPLATE.format(
        documents=format_documents(documents_raw),
        question=question,
        response=response,
    )
    completion = client.chat.completions.create(
        model=MODEL,
        messages=[{"role": "user", "content": prompt}],
        temperature=1.0,
        max_tokens=1024,
    )
    return completion.choices[0].message.content.strip()


print("Helpers defined ✓")

In [ ]:
# ── Main loop ─────────────────────────────────────────────────────────────
results = []

for idx in range(N_ROWS):
    row = train[idx]

    documents = row.get("documents") or row.get("document") or row.get("context") or ""
    question  = row.get("question")  or row.get("query")    or ""
    response  = row.get("response")  or row.get("answer")   or row.get("output") or ""

    print(f"\n{'='*70}")
    print(f"ROW {idx+1} / {N_ROWS}")
    print(f"{'='*70}")
    print(f"\n📄 QUESTION:\n{question}")
    print(f"\n✅ ORIGINAL RESPONSE:\n{response}")

    corrupted = insert_error(documents, question, response)
    print(f"\n⚠️  CORRUPTED RESPONSE (tagged):\n{corrupted}")

    results.append({"idx": idx, "question": question, "response": response, "corrupted": corrupted})

print(f"\n\n✔ Done — {len(results)} rows processed.")

In [ ]:
# ── Sanity check ──────────────────────────────────────────────────────────
import re

TAG_PATTERN = re.compile(r"<(Temporal|Numerical|Entity|Relation|Contradictory|Unverifiable)>")

print(f"{'Row':<5} {'Tag found':<15} {'Error type detected'}")
print("-" * 42)
for r in results:
    match = TAG_PATTERN.search(r["corrupted"])
    etype = match.group(1) if match else "— NONE DETECTED —"
    print(f"{r['idx']+1:<5} {'✅' if match else '❌':<15} {etype}")